In [ ]:
import os
import re
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi


load_dotenv()

# print(os.getenv("DEFAULT_LLM_PROVIDER", "gemini"))
# print(os.getenv("DEFAULT_LLM_PROVIDER"))


def get_youtube_transcript(url_or_video_id: str) -> str:
    try:
        video_id = url_or_video_id.strip()
        match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11})", url_or_video_id)
        if match:
            video_id = match.group(1)

        api = YouTubeTranscriptApi()
        
        transcript_list_obj = api.list(video_id)
        
        first_transcript = next(iter(transcript_list_obj))
        
        transcript_data = first_transcript.fetch()

        lines = []
        for item in transcript_data:
            if hasattr(item, 'text'):
                lines.append(item.text)
            elif isinstance(item, dict) and 'text' in item:
                lines.append(item['text'])
            else:
                lines.append(str(item))

        full_transcript = " ".join(lines)
        if not full_transcript.strip():
            return f"Transcript for video '{video_id}' is empty."

        return full_transcript[:10000] + "\n...[cut]" if len(full_transcript) > 10000 else full_transcript

    except Exception as e:
        return f"Could not retrieve transcript for video ID '{video_id}'. Error: {str(e)}"



get_youtube_transcript("https://www.youtube.com/watch?v=FQl_iZHu9Q8")
